In [ ]:
import numpy as np
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset

from climate_attitudes.utils import pcorr

config = Config(_env_file="../.env")

data = Dataset.load(config)

Filter down to participants who are present in waves 1 and 2

In [ ]:
pids = (
    data.participant.filter(wave_1=True, wave_2=True)
    .select("participant_id")
    .collect()
    .to_series()
    .implode()
)

resp = (
    data.response.filter(pl.col("wave") <= 2, pl.col("participant_id").is_in(pids))
    .select(
        "participant_id",
        pl.col("wave").replace_strict({1: "wave_1", 2: "wave_2"}),
        pl.col("cc1").replace({1: 2, 99: 1}),
        "cvcc4_should",
        pl.col("cc5_world").replace(99, None),
        "cc6",
    )
    .filter(pl.all_horizontal(pl.all().is_not_null()))
    .with_columns(pl.len().over("participant_id").alias("n_waves"))
    .filter(n_waves=2)
    .drop("n_waves")
    .with_columns(
        pl.col("cc1") / 2,
        (pl.col("cvcc4_should") - 1) / 4,
        (pl.col("cc5_world") - 1) / 3,
        (pl.col("cc6") - 1) / 3,
    )
    .collect()
)

columns = {
    "Climate change happening": {
        "colname": "cc1",
        "responses": ["No", "Maybe", "Yes"],
    },
    "Climate change anthropogenic ('people should act')": {
        "colname": "cvcc4_should",
        "responses": [
            "Strongly disagree",
            "Disagree",
            "Neither agree nor disagree",
            "Agree",
            "Strongly agree",
        ],
    },
    "Climate change worry": {
        "colname": "cc6",
        "responses": [
            "Not at all worried",
            "Not very worried",
            "Somewhat worried",
            "Very worried",
        ],
    },
    "Future generation harm": {
        "colname": "cc5_world",
        "responses": [
            "Don't know",
            "Not at all",
            "Only a little",
            "A moderate amount",
            "A great deal",
        ],
    },
}

In [ ]:
corr = resp.drop("participant_id", "wave").to_pandas().corr()

partial_corr = pcorr(
    resp
    # .filter(wave="wave_2")
    .drop("participant_id", "wave")
)

# # Generate a mask for the upper triangle
mask = np.triu(np.ones_like(partial_corr, dtype=bool), k=1)

# Set up the matplotlib figure
fig, axes = plt.subplots(ncols=2, figsize=(9, 6), sharey=True, constrained_layout=True)

# Generate a custom diverging colormap
cmap = sns.diverging_palette(230, 20, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    corr,
    mask=mask,
    cmap=cmap,
    annot=True,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar=False,
    # cbar_kws={"shrink": 0.5},
    ax=axes[0],
)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(
    partial_corr,
    mask=mask,
    cmap=cmap,
    annot=True,
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.5},
    ax=axes[1],
)

labels = ["CC Happening", "CC Anthropogenic (proxy)", "CC Worry", "Future gen harm"]

axes[1].set_yticks([])
axes[0].set_yticks(np.arange(4) + 0.5, labels=labels, rotation=0)

for axe in axes.flatten():
    axe.set_xticks(
        np.arange(4) + 0.5, labels=labels, rotation=30, horizontalalignment="right"
    )


axes[0].set_title("Correlation")
axes[1].set_title("Partial correlation");